In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn

Gli **Sparse Autoencoders (SAE)** implementano una forma di *sparse dictionary learning*, con l'obiettivo di apprendere una decomposizione sparsa di un segnale in un dizionario sovraccompleto di atomi.

---

Un SAE è costituito da:
- **Encoder** $W_{enc} \in \mathbb{R}^{d \times \omega}$: trasforma l'embedding di input in uno spazio latente
- **Decoder** $W_{dec} \in \mathbb{R}^{\omega \times d}$: ricostruisce l'embedding originale dallo spazio latente
- **Funzione di attivazione non-lineare** $\sigma : \mathbb{R}^{\omega} \to \mathbb{R}^{\omega}$
- **Bias condiviso** $b \in \mathbb{R}^d$: sottratto dall'input dell'encoder e aggiunto all'output del decoder

La larghezza dello strato latente $\omega$ è scelta come fattore della dimensione originale: $\omega := d \times \varepsilon$, dove $\varepsilon$ è il **fattore di espansione**.

---

Dato un embedding $v \in \mathbb{R}^d$, il SAE decompose il vettore in:
- **Vettore di attivazioni**: $\phi(v) := \sigma(W_{enc}^{\top}(v - b))$
- **Vettore ricostruito**: $\hat{v} := W_{dec}^{\top}\phi(v) + b$

---

La loss function combina un **obiettivo di ricostruzione** con una **regolarizzazione di sparsità**:

$$\mathcal{L}(v) = R(v) + \lambda S(v)$$

dove:
- **Ricostruzione (L2)**: $R(v) := \|v - \hat{v}\|_2^2$ garantisce la fedeltà dell'informazione
- **Sparsità (L1)**: $S(v) := \|\phi(v)\|_1$ penalizza l'attivazione di troppi neuroni latenti
- **Hyperparameter** $\lambda$: regola il trade-off tra ricostruzione e sparsità

Nel nostro modello, utilizziamo $\sigma(\cdot) := \text{ReLU}(\cdot)$ come funzione di attivazione, garantendo che le attivazioni latenti siano non-negative e genuinamente sparse.

In [2]:
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim: int = 512, hidden_dim: int = 2048):
        super(SparseAutoencoder, self).__init__()
        self.b_dec=nn.Parameter(torch.zeros(input_dim))
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim,bias=False)
    def normalize_decoder_weights(self):
        with torch.no_grad():
            self.decoder.weight.data = F.normalize(self.decoder.weight.data, p=2, dim=0)

    def forward(self, x: torch.Tensor):
        x_centered= x-self.b_dec
        z = F.relu(self.encoder(x_centered))
        x_hat = self.decoder(z) + self.b_dec
        
        return x_hat, z

def sae_loss_function(x: torch.Tensor, x_hat: torch.Tensor, z: torch.Tensor, l1_lambda: float = 1e-4):
    mse_loss = F.mse_loss(x_hat, x)
    l1_loss = z.abs().mean()
    total_loss = mse_loss + l1_lambda * l1_loss
    
    return total_loss, mse_loss, l1_loss

sae = SparseAutoencoder(input_dim=512, hidden_dim=2048)
print(sae)

SparseAutoencoder(
  (encoder): Linear(in_features=512, out_features=2048, bias=True)
  (decoder): Linear(in_features=2048, out_features=512, bias=False)
)


In [4]:
#%pip install transformers pillow
import torch
from transformers import AutoProcessor, AutoModel
from PIL import Image
import torch.nn.functional as F


In [5]:
def l2_normalize(embeddings: torch.Tensor) -> torch.Tensor:
    # F.normalize divide ogni vettore per la sua norma L2
    return F.normalize(embeddings, p=2, dim=1)

In [ ]:
model_id = "openai/clip-vit-base-patch32"

print(f"Scaricamento del modello {model_id} in corso...")
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

model.eval()
image = Image.open("../images/raggi_x.jpg")
inputs = processor(images=image, return_tensors="pt")

with torch.no_grad():
    # Il modello restituisce l'oggetto contenitore
    vision_outputs = model.get_image_features(**inputs)
    
    # Estraiamo il tensore puro che rappresenta l'immagine intera
    vision_tensor = vision_outputs.pooler_output

vision_embeddings = F.normalize(vision_tensor, p=2, dim=1)

print("Shape dell'embedding estratto:", vision_embeddings.shape)

Scaricamento del modello openai/clip-vit-base-patch32 in corso...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Shape dell'embedding estratto: torch.Size([1, 512])


In [13]:
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

dataset= TensorDataset(vision_embeddings)
batch_size=64
dataloader=DataLoader(dataset,batch_size=batch_size,shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizzando il device: {device}")

sae = SparseAutoencoder(input_dim=512, hidden_dim=2048).to(device)
learning_rate = 1e-3
optimizer = optim.Adam(sae.parameters(), lr=learning_rate)

l1_lambda = 1e-4
num_epochs = 20

for epoch in range(num_epochs):
    sae.train()
    epoch_total_loss=0.0
    epoch_mse_loss=0.0
    epoch_l1_loss=0.0

    for batch in dataloader:
        x=batch[0].to(device)
        #forward 
        x_hat,z=sae(x)

        #calcolo loss
        total_loss, mse_loss, l1_loss = sae_loss_function(x, x_hat, z, l1_lambda)
        #backward e ottimizzazione
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        sae.normalize_decoder_weights() #si forza ad 1 per evitare il collasso

        #aggiorniamo le metriche
        epoch_total_loss += total_loss.item()
        epoch_mse_loss += mse_loss.item()
        epoch_l1_loss += l1_loss.item()
    
    avg_total_loss = epoch_total_loss / len(dataloader)
    avg_mse = epoch_mse_loss / len(dataloader)
    avg_l1 = epoch_l1_loss / len(dataloader)

    if (epoch + 1) % 5 == 0:
        print(f"Epoca [{epoch+1:02d}/{num_epochs}] | "
              f"Loss Tot: {avg_total_loss:.4f} | "
              f"MSE (Ricostruzione): {avg_mse:.4f} | "
              f"L1 (Sparsità): {avg_l1:.4f}")

Utilizzando il device: cpu
Epoca [05/20] | Loss Tot: 0.0005 | MSE (Ricostruzione): 0.0005 | L1 (Sparsità): 0.0055
Epoca [10/20] | Loss Tot: 0.0006 | MSE (Ricostruzione): 0.0006 | L1 (Sparsità): 0.0042
Epoca [15/20] | Loss Tot: 0.0002 | MSE (Ricostruzione): 0.0002 | L1 (Sparsità): 0.0050
Epoca [20/20] | Loss Tot: 0.0001 | MSE (Ricostruzione): 0.0001 | L1 (Sparsità): 0.0050




L'addestramento non supervisionato del nostro *Sparse Autoencoder* (SAE) ha prodotto un dizionario sovracompleto di feature: la matrice dei pesi del decoder, $W_{dec} \in \mathbb{R}^{512 \times 2048}$. Ogni colonna di questa matrice rappresenta un singolo "concetto visivo" che il modello ha isolato autonomamente guardando i pixel. Tuttavia, questi concetti sono matematicamente "muti": il modello ne riconosce l'esistenza, ma non possiede un'etichetta semantica umana per descriverli.

L'assegnazione delle etichette (Grounding) avviene confrontando i concetti del SAE con gli embedding testuali. Poiché entrambi i tensori sono stati preventivamente assoggettati a **normalizzazione L2** (rendendo i vettori di norma unitaria), il calcolo della similarità coseno si riduce a una singola moltiplicazione matriciale:

$$S = T \cdot W_{dec}$$

Dove $T$ è la matrice degli embedding testuali e $W_{dec}$ è il dizionario del SAE. Estraendo i valori massimi (*Top-K*) da questo prodotto, identifichiamo in modo inequivocabile quali specifici neuroni dell'Autoencoder si sono specializzati nel rilevare determinati concetti medici, rendendo la rappresentazione finale totalmente interpretabile.

In [18]:
#i nostri concetti 
medical_concepts = [
    "healthy lungs", 
    "bone fracture", 
    "pneumonia", 
    "pleural effusion",
    "heart",
    "ribs",
    "medical imaging artifact"
]

text_inputs=processor(text=medical_concepts,padding=True,return_tensors='pt')

with torch.no_grad():
    text_outputs=model.get_text_features(**text_inputs)
    text_tensor = text_outputs.pooler_output
#normalizziamo i concetti (N_concetti, 512)
text_embeddings=F.normalize(text_tensor, p=2,dim=1).to(device)

with torch.no_grad():
    sae_dictionary=F.normalize(sae.decoder.weight.data,p=2,dim=0)  

similarities=torch.matmul(text_embeddings,sae_dictionary) #calcoliamo la similarità 

top_k = 3
for idx, concept in enumerate(medical_concepts):
    concept_sims = similarities[idx]
    # Otteniamo i primi k valori e i loro indici (i "neuroni" del SAE)
    top_values, top_indices = torch.topk(concept_sims, top_k)
    
    print(f"\nConcetto testuale: '{concept}'")
    for i in range(top_k):
        print(f"  -> Neurone SAE {top_indices[i].item():4d} (Similarità: {top_values[i].item():.4f})")


Concetto testuale: 'healthy lungs'
  -> Neurone SAE  505 (Similarità: 0.1308)
  -> Neurone SAE  116 (Similarità: 0.1242)
  -> Neurone SAE 1687 (Similarità: 0.1225)

Concetto testuale: 'bone fracture'
  -> Neurone SAE 1479 (Similarità: 0.1476)
  -> Neurone SAE 1297 (Similarità: 0.1401)
  -> Neurone SAE 1350 (Similarità: 0.1265)

Concetto testuale: 'pneumonia'
  -> Neurone SAE 1479 (Similarità: 0.1325)
  -> Neurone SAE  892 (Similarità: 0.1266)
  -> Neurone SAE 1932 (Similarità: 0.1249)

Concetto testuale: 'pleural effusion'
  -> Neurone SAE 1292 (Similarità: 0.1404)
  -> Neurone SAE 1768 (Similarità: 0.1261)
  -> Neurone SAE  266 (Similarità: 0.1248)

Concetto testuale: 'heart'
  -> Neurone SAE 1687 (Similarità: 0.1331)
  -> Neurone SAE 1253 (Similarità: 0.1264)
  -> Neurone SAE  109 (Similarità: 0.1251)

Concetto testuale: 'ribs'
  -> Neurone SAE 1687 (Similarità: 0.1455)
  -> Neurone SAE 1354 (Similarità: 0.1362)
  -> Neurone SAE  272 (Similarità: 0.1350)

Concetto testuale: 'medical